<a href="https://colab.research.google.com/github/duongduyvinh/Speech-Emote-Recognition/blob/main/soft-vc-demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import json

# 1. Định nghĩa đường dẫn tới file notebook của bạn (sửa tên file nếu bạn đặt tên khác)
notebook_path = "/content/drive/MyDrive/Colab Notebooks/soft-vc-demo.ipynb"

try:
    with open(notebook_path, "r", encoding="utf-8") as f:
        nb_data = json.load(f)

    # 2. Xóa bỏ cấu trúc metadata widgets gây crash trình render của GitHub
    if "widgets" in nb_data.get("metadata", {}):
        del nb_data["metadata"]["widgets"]

    with open(notebook_path, "w", encoding="utf-8") as f:
        json.dump(nb_data, f, indent=1, ensure_ascii=False)

    print("✨ Đã dọn sạch Metadata Widget lỗi thành công! Bây giờ bạn hãy bấm tiến hành Commit và Push lại lên GitHub nhé.")
except Exception as e:
    print(f"❌ Không tìm thấy file tại đường dẫn chỉ định: {e}")

❌ Không tìm thấy file tại đường dẫn chỉ định: [Errno 2] No such file or directory: '/content/drive/MyDrive/Colab Notebooks/soft-vc-demo.ipynb'


# 🛠️ THIẾT LẬP MÔI TRƯỜNG CHUYÊN GIA (ENV SETUP)
Bước này chuẩn bị GPU, thư viện và đường dẫn dữ liệu cho pipeline SER.

In [7]:
import torch
import torchaudio
import os
import glob
from google.colab import drive

# 1. Khởi động GPU
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"🖥️ Thiết bị: {device.upper()}")
if device == "cuda":
    print(f"🚀 GPU Type: {torch.cuda.get_device_name(0)}")

# 2. Mount Drive (nếu chưa)
if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')

# 3. Tạo thư mục làm việc tốc độ cao trên Local Disk (SSD của Colab)
!mkdir -p ./data/ravdess
!mkdir -p ./data/synthetic_wavs

print("✅ Môi trường đã sẵn sàng.")

🖥️ Thiết bị: CUDA
🚀 GPU Type: Tesla T4
✅ Môi trường đã sẵn sàng.


In [8]:
# 4. Nạp model Soft-VC lên GPU (Kiểm tra lại để chắc chắn không bị giải phóng bộ nhớ)
try:
    hubert = torch.hub.load("bshall/hubert:main", "hubert_soft", trust_repo=True).to(device)
    acoustic = torch.hub.load("bshall/acoustic-model:main", "hubert_soft", trust_repo=True).to(device)
    hifigan = torch.hub.load("bshall/hifigan:main", "hifigan_hubert_soft", trust_repo=True).to(device)
    hubert.eval(); acoustic.eval(); hifigan.eval()
    print("✅ Bộ ba model Soft-VC đã nạp vào GPU sẵn sàng!")
except Exception as e:
    print(f"❌ Lỗi nạp model: {e}")

Downloading: "https://github.com/bshall/hubert/zipball/main" to /root/.cache/torch/hub/main.zip
Downloading: "https://github.com/bshall/hubert/releases/download/v0.2/hubert-soft-35d9f29f.pt" to /root/.cache/torch/hub/checkpoints/hubert-soft-35d9f29f.pt


100%|██████████| 361M/361M [00:12<00:00, 30.0MB/s]


Downloading: "https://github.com/bshall/acoustic-model/zipball/main" to /root/.cache/torch/hub/main.zip
Downloading: "https://github.com/bshall/acoustic-model/releases/download/v0.1/hubert-soft-0321fd7e.pt" to /root/.cache/torch/hub/checkpoints/hubert-soft-0321fd7e.pt


100%|██████████| 71.8M/71.8M [00:02<00:00, 37.6MB/s]


Downloading: "https://github.com/bshall/hifigan/zipball/main" to /root/.cache/torch/hub/main.zip


/usr/local/lib/python3.12/dist-packages/torch/nn/utils/weight_norm.py:144: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)


Downloading: "https://github.com/bshall/hifigan/releases/download/v0.1/hifigan-hubert-soft-65f03469.pt" to /root/.cache/torch/hub/checkpoints/hifigan-hubert-soft-65f03469.pt


100%|██████████| 54.9M/54.9M [00:02<00:00, 25.7MB/s]

✅ Bộ ba model Soft-VC đã nạp vào GPU sẵn sàng!


# 🚀 KHỞI ĐỘNG LẠI DỰ ÁN VỚI GPU T4
Quy trình này đã được tối ưu hóa để chạy trên NVIDIA T4.

In [9]:
import torch
import torchaudio
import os
import requests

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"🔥 Đang chạy trên: {device.upper()}")

# Tải model trực tiếp lên GPU
try:
    hubert = torch.hub.load("bshall/hubert:main", "hubert_soft", trust_repo=True).to(device)
    acoustic = torch.hub.load("bshall/acoustic-model:main", "hubert_soft", trust_repo=True).to(device)
    hifigan = torch.hub.load("bshall/hifigan:main", "hifigan_hubert_soft", trust_repo=True).to(device)
    print("✅ Các model Soft-VC đã sẵn sàng trên GPU!")
except Exception as e:
    print(f"❌ Lỗi: {e}")

🔥 Đang chạy trên: CUDA


Using cache found in /root/.cache/torch/hub/bshall_hubert_main
Using cache found in /root/.cache/torch/hub/bshall_acoustic-model_main
Using cache found in /root/.cache/torch/hub/bshall_hifigan_main
/usr/local/lib/python3.12/dist-packages/torch/nn/utils/weight_norm.py:144: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)


✅ Các model Soft-VC đã sẵn sàng trên GPU!


In [24]:
import glob
import os

# Tìm file audio đầu tiên trong tập dữ liệu RAVDESS thực tế của bạn
# Đường dẫn sau khi giải nén thường là ./data/ravdess/audio_speech_actors_01-24/...
ravdess_files = glob.glob("./data/ravdess/**/*.wav", recursive=True)

if ravdess_files:
    test_input = ravdess_files[0]
    print(f"✅ Đã tìm thấy dữ liệu thật: {test_input}")
    os.environ['TEST_WAV'] = test_input
else:
    print("❌ Cảnh báo: Thư mục ./data/ravdess/ đang trống.")
    print("💡 Hãy chắc chắn bạn đã chạy ô giải nén (cell u6Jfn3ZxE-FW) trước đó.")

✅ Đã tìm thấy dữ liệu thật: ./data/ravdess/Actor_19/03-01-06-02-01-01-19.wav


In [25]:
def convert_voice_gpu(source_path, output_path):
    try:
        # Load audio với torchaudio
        source, sr = torchaudio.load(source_path)
        source = torchaudio.functional.resample(source, sr, 16000)
        if source.shape[0] > 1: source = source.mean(0, keepdim=True)
        source = source.unsqueeze(0).to(device)

        # Xử lý bẻ giọng siêu tốc trên T4 GPU
        with torch.inference_mode():
            units = hubert.units(source)
            mel = acoustic.generate(units).transpose(1, 2)
            target = hifigan(mel)

        # Lưu kết quả
        torchaudio.save(output_path, target.squeeze(0).cpu(), 16000)
        print(f"✨ THÀNH CÔNG! Đã bẻ giọng file: {source_path}")
        return True
    except Exception as e:
        print(f"❌ Lỗi xử lý file {source_path}: {e}")
        return False

# Chạy thử trên dữ liệu thật vừa tìm được
test_file = os.environ.get('TEST_WAV')
if test_file and os.path.exists(test_file):
    convert_voice_gpu(test_file, "ravdess_test_output.wav")
else:
    print("⚠️ Không có file đầu vào. Hãy chạy lại cell tìm file ở trên.")

✨ THÀNH CÔNG! Đã bẻ giọng file: ./data/ravdess/Actor_19/03-01-06-02-01-01-19.wav


In [26]:
import glob
from tqdm import tqdm
import os

def run_emotion_conversion_full_8(root_path, output_dir):
    # Danh sách đầy đủ 8 mã cảm xúc của RAVDESS
    # 01=neutral, 02=calm, 03=happy, 04=sad, 05=angry, 06=fearful, 07=disgust, 08=surprised
    emotions = {
        "01": "Neutral", "02": "Calm", "03": "Happy", "04": "Sad",
        "05": "Angry", "06": "Fearful", "07": "Disgust", "08": "Surprised"
    }

    # Lấy danh sách Neutral làm gốc để bẻ giọng
    neutral_files = glob.glob(os.path.join(root_path, "**", "03-01-01-*.wav"), recursive=True)
    print(f"🔍 Tìm thấy {len(neutral_files)} file Neutral gốc. Bắt đầu sinh 8 lớp...")

    os.makedirs(output_dir, exist_ok=True)

    for src_path in tqdm(neutral_files, desc="Sinh dữ liệu 8 lớp"):
        file_name = os.path.basename(src_path)
        parts = file_name.split("-")

        for emo_code, emo_name in emotions.items():
            new_parts = parts.copy()
            new_parts[2] = emo_code
            new_name = f"syn_{emo_name}_{'-'.join(new_parts)}"
            out_path = os.path.join(output_dir, new_name)

            if not os.path.exists(out_path):
                convert_voice_gpu(src_path, out_path)

# Thực thi sinh toàn bộ 8 lớp
ravdess_local = "./data/ravdess"
synthetic_dir = "/content/drive/MyDrive/ResLSTM-SER/data/synthetic_wavs"
run_emotion_conversion_full_8(ravdess_local, synthetic_dir)

🔍 Tìm thấy 96 file Neutral gốc. Bắt đầu sinh 8 lớp...


Sinh dữ liệu 8 lớp: 100%|██████████| 96/96 [00:00<00:00, 574.45it/s]


In [27]:
import os
import glob

synthetic_path = "/content/drive/MyDrive/ResLSTM-SER/data/synthetic_wavs"
files = glob.glob(os.path.join(synthetic_path, "*.wav"))

print(f"📊 Tổng số file nhân tạo hiện có trên Drive: {len(files)}")
if len(files) >= 384:
    print("✨ Đã hoàn thành mục tiêu tạo dữ liệu (192 Neutral -> 192 Happy + 192 Angry).")
else:
    print(f"⏳ Vẫn đang trong quá trình xử lý hoặc đồng bộ... ({len(files)}/384)")

📊 Tổng số file nhân tạo hiện có trên Drive: 768
✨ Đã hoàn thành mục tiêu tạo dữ liệu (192 Neutral -> 192 Happy + 192 Angry).


In [28]:
from transformers import Wav2Vec2Model, Wav2Vec2Processor
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"

print("⏳ Đang nạp Wav2Vec 2.0 Large để chuẩn bị trích xuất đặc trưng...")
model_name = "facebook/wav2vec2-large-960h"

try:
    processor = Wav2Vec2Processor.from_pretrained(model_name)
    wav2vec_model = Wav2Vec2Model.from_pretrained(model_name).to(device)
    wav2vec_model.eval()
    print(f"✅ Đã nạp thành công Wav2Vec 2.0 Large lên {device.upper()}!")
    print(f"📐 Kích thước Hidden State: {wav2vec_model.config.hidden_size} (1024 chiều)")
except Exception as e:
    print(f"❌ Lỗi nạp mô hình: {e}")

⏳ Đang nạp Wav2Vec 2.0 Large để chuẩn bị trích xuất đặc trưng...


preprocessor_config.json:   0%|          | 0.00/159 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/843 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/163 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/291 [00:00<?, ?B/s]

...

pytorch_model.bin:   0%|          | 0.00/1.26G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/402 [00:00<?, ?it/s]

[transformers] Wav2Vec2Model LOAD REPORT from: facebook/wav2vec2-large-960h
Key               | Status     | 
------------------+------------+-
lm_head.weight    | UNEXPECTED | 
lm_head.bias      | UNEXPECTED | 
masked_spec_embed | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


model.safetensors:   0%|          | 0.00/1.26G [00:00<?, ?B/s]

✅ Đã nạp thành công Wav2Vec 2.0 Large lên CUDA!
📐 Kích thước Hidden State: 1024 (1024 chiều)


In [29]:
import torch
import torchaudio
import os
from tqdm import tqdm

def extract_wav2vec_features(file_path, output_path):
    try:
        # 1. Load và resample về 16kHz chuẩn cho Wav2Vec
        waveform, sample_rate = torchaudio.load(file_path)
        if sample_rate != 16000:
            waveform = torchaudio.functional.resample(waveform, sample_rate, 16000)

        # Chuyển sang mono nếu cần
        if waveform.shape[0] > 1:
            waveform = torch.mean(waveform, dim=0, keepdim=True)

        # 2. Xử lý qua processor và model
        inputs = processor(waveform.squeeze().numpy(), sampling_rate=16000, return_tensors="pt", padding=True)
        input_values = inputs.input_values.to(device)

        with torch.no_grad():
            outputs = wav2vec_model(input_values)
            # Lấy hidden states cuối cùng (Batch, Time, 1024)
            embeddings = outputs.last_hidden_state.cpu()

        # 3. Lưu tensor đặc trưng
        torch.save(embeddings, output_path)
        return True
    except Exception as e:
        print(f"❌ Lỗi trích xuất {file_path}: {e}")
        return False

# Thiết lập đường dẫn lưu trữ embeddings
features_dir = "/content/drive/MyDrive/ResLSTM-SER/data/features"
os.makedirs(features_dir, exist_ok=True)

# Thử trích xuất cho 1 file mẫu trước khi chạy batch
if len(files) > 0:
    sample_file = files[0]
    sample_out = os.path.join(features_dir, os.path.basename(sample_file).replace(".wav", ".pt"))
    if extract_wav2vec_features(sample_file, sample_out):
        print(f"✅ Đã trích xuất đặc trưng mẫu thành công: {sample_out}")
        print(f"📐 Kích thước tensor: {torch.load(sample_out).shape}")

✅ Đã trích xuất đặc trưng mẫu thành công: /content/drive/MyDrive/ResLSTM-SER/data/features/syn_Happy_03-01-03-01-01-02-19.pt
📐 Kích thước tensor: torch.Size([1, 160, 1024])


In [30]:
# Thực hiện trích xuất đặc trưng hàng loạt cho toàn bộ file hiện có
print(f"🚀 Bắt đầu trích xuất đặc trưng cho {len(files)} file...")

success_count = 0
for wav_file in tqdm(files, desc="Feature Extraction"):
    file_name = os.path.basename(wav_file)
    output_name = file_name.replace(".wav", ".pt")
    output_path = os.path.join(features_dir, output_name)

    # Kiểm tra nếu file đã tồn tại thì bỏ qua
    if not os.path.exists(output_path):
        if extract_wav2vec_features(wav_file, output_path):
            success_count += 1
    else:
        success_count += 1

print(f"\n✅ Hoàn thành! Đã xử lý thành công {success_count}/{len(files)} file.")
print(f"📂 Các tệp đặc trưng (.pt) được lưu tại: {features_dir}")

🚀 Bắt đầu trích xuất đặc trưng cho 768 file...


Feature Extraction: 100%|██████████| 768/768 [00:00<00:00, 2254.08it/s]


✅ Hoàn thành! Đã xử lý thành công 768/768 file.
📂 Các tệp đặc trưng (.pt) được lưu tại: /content/drive/MyDrive/ResLSTM-SER/data/features


In [31]:
import os
import glob

synthetic_path = "/content/drive/MyDrive/ResLSTM-SER/data/synthetic_wavs"
files = glob.glob(os.path.join(synthetic_path, "*.wav"))

print(f"📊 Tổng số file nhân tạo hiện có trên Drive: {len(files)}")
if len(files) >= 384:
    print("✨ Đã hoàn thành mục tiêu tạo dữ liệu (192 Neutral -> 192 Happy + 192 Angry).")
else:
    print(f"⏳ Vẫn đang trong quá trình đồng bộ hoặc xử lý... ({len(files)}/384)")

📊 Tổng số file nhân tạo hiện có trên Drive: 768
✨ Đã hoàn thành mục tiêu tạo dữ liệu (192 Neutral -> 192 Happy + 192 Angry).


In [35]:
from transformers import Wav2Vec2Model, Wav2Vec2Processor
import torch

# Sử dụng GPU T4 đã được cấu hình trước đó
device = "cuda" if torch.cuda.is_available() else "cpu"

print("⏳ Đang nạp Wav2Vec 2.0 Large để chuẩn bị trích xuất đặc trưng...")
model_name = "facebook/wav2vec2-large-960h"

try:
    processor = Wav2Vec2Processor.from_pretrained(model_name)
    wav2vec_model = Wav2Vec2Model.from_pretrained(model_name).to(device)
    wav2vec_model.eval()
    print(f"✅ Đã nạp thành công Wav2Vec 2.0 Large lên {device.upper()}!")
    print(f"📐 Kích thước Hidden State: {wav2vec_model.config.hidden_size} (Khớp chuẩn 1024 chiều cho ResLSTM)")
except Exception as e:
    print(f"❌ Lỗi nạp mô hình: {e}")

⏳ Đang nạp Wav2Vec 2.0 Large để chuẩn bị trích xuất đặc trưng...


Loading weights:   0%|          | 0/402 [00:00<?, ?it/s]

[transformers] Wav2Vec2Model LOAD REPORT from: facebook/wav2vec2-large-960h
Key               | Status     | 
------------------+------------+-
lm_head.weight    | UNEXPECTED | 
lm_head.bias      | UNEXPECTED | 
masked_spec_embed | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


✅ Đã nạp thành công Wav2Vec 2.0 Large lên CUDA!
📐 Kích thước Hidden State: 1024 (Khớp chuẩn 1024 chiều cho ResLSTM)


In [36]:
import os
import glob

synthetic_path = "/content/drive/MyDrive/ResLSTM-SER/data/synthetic_wavs"
files = glob.glob(os.path.join(synthetic_path, "*.wav"))

print(f"📊 Tổng số file nhân tạo đã tạo: {len(files)}")
if len(files) > 0:
    print("✅ Dữ liệu đã sẵn sàng để trích xuất đặc trưng Wav2Vec 2.0.")
else:
    print("⚠️ Cảnh báo: Thư mục hiện tại đang trống. Vui lòng đợi script bẻ giọng hoàn tất.")

📊 Tổng số file nhân tạo đã tạo: 768
✅ Dữ liệu đã sẵn sàng để trích xuất đặc trưng Wav2Vec 2.0.


### 🛠️ Quy trình Trích xuất Đặc trưng Hàng loạt (Batch Feature Extraction)

Trong bước này, chúng ta sẽ chuyển đổi toàn bộ các tệp âm thanh (.wav) đã được bẻ giọng thành các vector đặc trưng số học (.pt).

**Tại sao dùng Wav2Vec 2.0 Large?**
*   **Hidden State (1024-dim):** Mô hình này trích xuất ngữ cảnh âm thanh sâu sắc với 1024 chiều, cung cấp đầu vào cực kỳ chất lượng cho mô hình phân loại **ResLSTM** của bạn.
*   **Chuẩn hóa:** Toàn bộ âm thanh sẽ được đưa về tần số lấy mẫu 16kHz để khớp với yêu cầu của mô hình pre-trained.
*   **Tối ưu hóa:** Mã nguồn dưới đây sẽ kiểm tra nếu tệp đặc trưng đã tồn tại trên Drive thì sẽ bỏ qua, giúp tiết kiệm thời gian nếu quá trình bị ngắt quãng.

In [34]:
import os
import torch
import glob
from tqdm import tqdm

# Cập nhật lại danh sách file sau khi sinh đủ 8 lớp
synthetic_path = "/content/drive/MyDrive/ResLSTM-SER/data/synthetic_wavs"
features_dir = "/content/drive/MyDrive/ResLSTM-SER/data/features"
os.makedirs(features_dir, exist_ok=True)

wav_files = glob.glob(os.path.join(synthetic_path, "*.wav"))
print(f"🔍 Tổng số file cần trích xuất đặc trưng: {len(wav_files)}")

success_count = 0
skip_count = 0

for wav_file in tqdm(wav_files, desc="Trích xuất Wav2Vec Features"):
    file_name = os.path.basename(wav_file)
    output_name = file_name.replace(".wav", ".pt")
    output_path = os.path.join(features_dir, output_name)

    if os.path.exists(output_path):
        skip_count += 1
        continue

    if extract_wav2vec_features(wav_file, output_path):
        success_count += 1

print(f"✅ Hoàn thành! Mới: {success_count}, Bỏ qua: {skip_count}")

🔍 Tổng số file cần trích xuất đặc trưng: 768


Trích xuất Wav2Vec Features: 100%|██████████| 768/768 [00:00<00:00, 3931.21it/s]

✅ Hoàn thành! Mới: 0, Bỏ qua: 768


### 📊 Kiểm tra kết quả
Sau khi chạy xong, chúng ta cần kiểm tra xem các tệp `.pt` đã được tạo đúng cấu trúc hay chưa. Mỗi tệp sẽ chứa một tensor có kích thước `[1, Time_Steps, 1024]`.

In [22]:
import torch

# Lấy thử một file vừa tạo để kiểm tra
feature_files = glob.glob(os.path.join(features_dir, "*.pt"))

if feature_files:
    sample_pt = feature_files[0]
    data = torch.load(sample_pt)
    print(f"📄 File kiểm tra: {os.path.basename(sample_pt)}")
    print(f"📐 Kích thước Tensor: {data.shape}")
    print(f"✅ Trạng thái: Sẵn sàng cho huấn luyện ResLSTM.")
else:
    print("⚠️ Chưa tìm thấy file đặc trưng nào. Hãy chạy cell xử lý ở trên.")

⚠️ Chưa tìm thấy file đặc trưng nào. Hãy chạy cell xử lý ở trên.


### 📦 Xây dựng Dataset và DataLoader cho ResLSTM

Trong bước này, chúng ta sẽ:
1.  **EmotionDataset**: Đọc các tệp `.pt` từ Drive và gán nhãn cảm xúc dựa tràn tén file.
2.  **Collate Function**: Xử lý padding bằng `pad_sequence` để các tệp có độ dài khác nhau có thể đưa vào cùng một batch.
3.  **DataLoader**: Tạo các luồng dữ liệu song song.

In [23]:
import torch
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
import os
import glob

class EmotionDataset(Dataset):
    def __init__(self, features_dir):
        self.feature_files = glob.glob(os.path.join(features_dir, "*.pt"))
        # Cập nhật đầy đủ 8 nhãn RAVDESS (trừ 1 để index từ 0-7)
        # 01=neutral, 02=calm, 03=happy, 04=sad, 05=angry, 06=fearful, 07=disgust, 08=surprised

    def __len__(self):
        return len(self.feature_files)

    def __getitem__(self, idx):
        file_path = self.feature_files[idx]
        file_name = os.path.basename(file_path)

        # Trích xuất mã cảm xúc (phần thứ 3 trong tên file chuẩn RAVDESS)
        # Ví dụ: syn_Happy_03-01-03-... -> lấy '03'
        parts = file_name.split('-')
        if "syn" in parts[0]:
            emo_code = parts[2]
        else:
            emo_code = parts[2]

        label = int(emo_code) - 1 # Chuyển 01-08 thành 0-7

        feature = torch.load(file_path).squeeze(0)
        return feature, torch.tensor(label)

def collate_fn(batch):
    features, labels = zip(*batch)
    lengths = torch.tensor([f.shape[0] for f in features])
    features_padded = pad_sequence(features, batch_first=True, padding_value=0.0)
    return features_padded, torch.tensor(labels), lengths

# Khởi tạo lại với 8 lớp
dataset = EmotionDataset(features_dir)
train_loader = DataLoader(dataset, batch_size=16, shuffle=True, collate_fn=collate_fn)

print(f"✅ Đã cập nhật Dataset hỗ trợ 8 lớp cảm xúc. Tổng cộng: {len(dataset)} file.")

ValueError: num_samples should be a positive integer value, but got num_samples=0

### ✂️ Chia tập dữ liệu Train/Validation

Để đánh giá mô hình một cách khách quan, chúng ta sẽ chia 192 file hiện có thành hai phần:
- **Train Set (80%)**: Dùng để cập nhật trọng số mô hình.
- **Validation Set (20%)**: Dùng để kiểm tra độ chính xác trên dữ liệu chưa thấy.

In [ ]:
from torch.utils.data import DataLoader, WeightedRandomSampler, random_split
import numpy as np
import torch

# 1. Khởi tạo dataset mới chứa đủ 8 lớp
dataset = EmotionDataset(features_dir)

# 2. Tính toán trọng số cho WeightedRandomSampler
targets = []
for i in range(len(dataset)):
    _, label = dataset[i]
    targets.append(label.item())

targets = np.array(targets)
unique_classes, class_sample_count = np.unique(targets, return_counts=True)

class_weights = 1. / class_sample_count
weight_map = {cls: weight for cls, weight in zip(unique_classes, class_weights)}
samples_weight = torch.tensor([weight_map[t] for t in targets])

# 3. Chia tập dữ liệu
train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size
train_dataset, val_dataset = random_split(dataset, [train_size, val_size])

# 4. Tạo sampler cho train
train_indices = train_dataset.indices
train_sampler = WeightedRandomSampler(samples_weight[train_indices].type('torch.DoubleTensor'), len(train_indices))

# 5. Loader
train_loader = DataLoader(train_dataset, batch_size=32, sampler=train_sampler, collate_fn=collate_fn)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, collate_fn=collate_fn)

print(f"✅ Đã chuẩn bị dữ liệu 8 lớp: Train={len(train_dataset)}, Val={len(val_dataset)}")

### 🏋️ Khởi tạo Mô hình và Vòng lặp Huấn luyện

Bây giờ chúng ta sẽ khởi tạo mô hình `ResLSTM_Multi_Att` với đầu vào 1024 chiều từ Wav2Vec 2.0 và bắt đầu thiết lập Optimizer (Adam) cùng Loss Function (CrossEntropy).

In [ ]:
import sys
import os
import torch
import glob
import importlib.util

# 1. Đảm bảo mã nguồn đã được giải nén
extract_to = "/content/ResLSTM-SER"
if not os.path.exists(extract_to):
    zip_path = "/content/drive/MyDrive/ResLSTM-SER/ResLTSMv2.zip"
    !unzip -q "{zip_path}" -d {extract_to}

# 2. Tìm file res_lstm.py và nạp trực tiếp
search_results = glob.glob(f"{extract_to}/**/res_lstm.py", recursive=True)

if not search_results:
    print("❌ Không tìm thấy file res_lstm.py")
else:
    file_path = search_results[0]
    module_name = "res_lstm_module"

    spec = importlib.util.spec_from_file_location(module_name, file_path)
    res_lstm_module = importlib.util.module_from_spec(spec)
    sys.modules[module_name] = res_lstm_module

    try:
        spec.loader.exec_module(res_lstm_module)
        ResLSTM_Multi_Att = res_lstm_module.ResLSTM_Multi_Att

        device = "cuda" if torch.cuda.is_available() else "cpu"
        # Cập nhật dropout_rate theo yêu cầu (0.055)
        model = ResLSTM_Multi_Att(
            input_size=1024,
            hidden_size=64,
            num_layers=1,
            num_att=4,
            num_classes=8,
            device=device
        ).to(device)

        # Cập nhật LR (0.0086) và Weight Decay (0.0011) theo yêu cầu
        optimizer = torch.optim.Adam(model.parameters(), lr=0.008631730674354831, weight_decay=0.001163116600288255)
        criterion = torch.nn.CrossEntropyLoss()
        print(f"✅ Khởi tạo mô hình với LR mới: 0.0086 và Weight Decay: 0.0011")
    except Exception as e:
        print(f"❌ Lỗi: {e}")

### 📊 Bước 1: Kiểm tra sự cân bằng của 8 lớp cảm xúc
Trước khi huấn luyện, chúng ta cần xem trong 153 mẫu train và 39 mẫu val, mỗi cảm xúc có bao nhiêu đại diện.

In [ ]:
import collections

def check_balance(loader, name):
    all_labels = []
    for _, labels, _ in loader:
        all_labels.extend(labels.tolist())
    counter = collections.Counter(all_labels)
    print(f"\n📈 Phân bổ lớp trong {name}:")
    emotion_map = {0:'Neutral', 1:'Calm', 2:'Happy', 3:'Sad', 4:'Angry', 5:'Fearful', 6:'Disgust', 7:'Surprised'}
    for i in range(8):
        print(f" - {emotion_map[i]}: {counter[i]} mẫu")

check_balance(train_loader, "Tập Huấn luyện (Train)")
check_balance(val_loader, "Tập Kiểm định (Val)")

### 📉 Bước 2: Vòng lặp Huấn luyện cho 8 lớp cảm xúc
Chúng ta sẽ thực hiện huấn luyện và theo dõi độ chính xác (Accuracy) trên cả 8 lớp.

In [ ]:
from sklearn.metrics import recall_score

def train_model(model, train_loader, val_loader, epochs=100):
    history = {'train_loss': [], 'val_loss': [], 'val_acc': [], 'val_uar': []}
    best_uar = 0.0

    print(f"🚀 Bắt đầu huấn luyện {epochs} epoch...")

    for epoch in range(epochs):
        model.train()
        running_loss = 0.0

        for features, labels, lengths in train_loader:
            features, labels, lengths = features.to(device), labels.to(device), lengths.to(device)

            optimizer.zero_grad()
            logits, _ = model(features, lengths)
            loss = criterion(logits, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()

        # Validation phase
        model.eval()
        val_running_loss = 0.0
        all_preds = []
        all_labels = []

        with torch.no_grad():
            for features, labels, lengths in val_loader:
                features, labels, lengths = features.to(device), labels.to(device), lengths.to(device)
                logits, _ = model(features, lengths)
                loss = criterion(logits, labels)
                val_running_loss += loss.item()

                _, predicted = logits.max(1)
                all_preds.extend(predicted.cpu().numpy())
                all_labels.extend(labels.cpu().numpy())

        # Tính toán các chỉ số
        train_loss = running_loss / len(train_loader)
        val_loss = val_running_loss / len(val_loader)

        # UAR chuẩn = Unweighted Average Recall
        uar = recall_score(all_labels, all_preds, average='macro', zero_division=0) * 100
        # Accuracy chuẩn (Overall)
        acc = (np.array(all_preds) == np.array(all_labels)).mean() * 100

        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['val_acc'].append(acc)
        history['val_uar'].append(uar)

        if (epoch + 1) % 5 == 0 or epoch == 0:
            print(f"Epoch [{epoch+1}/{epochs}] | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Val UAR: {uar:.2f}%")

        if uar > best_uar:
            best_uar = uar
            torch.save(model.state_dict(), 'best_reslstm_model.pt')

    return history

if 'model' in locals():
    history = train_model(model, train_loader, val_loader, epochs=100)
else:
    print("❌ Lỗi: Không tìm thấy model.")

In [ ]:
import matplotlib.pyplot as plt

def plot_training_history(history):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

    # Biểu đồ Loss
    ax1.plot(history['train_loss'], label='Train Loss')
    ax1.plot(history['val_loss'], label='Val Loss')
    ax1.set_title('Model Loss History')
    ax1.set_xlabel('Epoch')
    ax1.set_ylabel('Loss')
    ax1.legend()
    ax1.grid(True)

    # Biểu đồ UAR
    ax2.plot(history['val_uar'], label='Val UAR (%)', color='orange')
    ax2.plot(history['val_acc'], label='Val Acc (%)', color='green', linestyle='--')
    ax2.set_title('Validation Metrics (UAR vs Accuracy)')
    ax2.set_xlabel('Epoch')
    ax2.set_ylabel('Percentage (%)')
    ax2.legend()
    ax2.grid(True)

    plt.show()

if 'history' in locals():
    plot_training_history(history)

### 🔄 Cập nhật Huấn luyện với K-Fold Cross-Validation

Dưới đây là quy trình huấn luyện chia theo Fold để đảm bảo tính khách quan và hiệu quả trên tập dữ liệu nhỏ.

In [ ]:
from sklearn.model_selection import KFold
from sklearn.metrics import recall_score
import numpy as np
import torch
from torch.utils.data import DataLoader, Subset
import os

# Khai báo lại các đường dẫn quan trọng để đảm bảo tính độc lập
features_dir = "/content/drive/MyDrive/ResLSTM-SER/data/features"

# Tạo thư mục nếu chưa tồn tại
os.makedirs(features_dir, exist_ok=True)

def train_with_kfold(dataset, n_splits=5, epochs=100):
    kfold = KFold(n_splits=n_splits, shuffle=True, random_state=42)
    fold_results = []
    indices = list(range(len(dataset)))

    for fold, (train_ids, val_ids) in enumerate(kfold.split(indices)):
        print(f"\n--- 📂 ĐANG HUẤN LUYỆN FOLD {fold + 1}/{n_splits} ---")

        train_sub = Subset(dataset, train_ids)
        val_sub = Subset(dataset, val_ids)

        train_loader = DataLoader(train_sub, batch_size=32, shuffle=True, collate_fn=collate_fn)
        val_loader = DataLoader(val_sub, batch_size=32, shuffle=False, collate_fn=collate_fn)

        # Reset model cho mỗi fold mới
        fold_model = ResLSTM_Multi_Att(
            input_size=1024, hidden_size=64, num_layers=1,
            num_att=4, num_classes=8, device=device
        ).to(device)

        fold_optimizer = torch.optim.Adam(
            fold_model.parameters(),
            lr=0.008631730674354831,
            weight_decay=0.001163116600288255
        )

        history = train_model_fold_optimized(fold_model, fold_optimizer, train_loader, val_loader, epochs=epochs)
        fold_results.append(history)
        plot_training_history(history)

    return fold_results

def train_model_fold_optimized(model, optimizer, train_loader, val_loader, epochs=100):
    history = {'train_loss': [], 'val_loss': [], 'val_acc': [], 'val_uar': []}
    criterion = torch.nn.CrossEntropyLoss()

    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        for features, labels, lengths in train_loader:
            features, labels, lengths = features.to(device), labels.to(device), lengths.to(device)
            optimizer.zero_grad()
            logits, _ = model(features, lengths)
            loss = criterion(logits, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()

        model.eval()
        val_running_loss = 0.0
        all_preds, all_labels = [], []
        with torch.no_grad():
            for features, labels, lengths in val_loader:
                features, labels, lengths = features.to(device), labels.to(device), lengths.to(device)
                logits, _ = model(features, lengths)
                val_running_loss += criterion(logits, labels).item()
                _, predicted = logits.max(1)
                all_preds.extend(predicted.cpu().numpy())
                all_labels.extend(labels.cpu().numpy())

        uar = recall_score(all_labels, all_preds, average='macro', zero_division=0) * 100
        acc = (np.array(all_preds) == np.array(all_labels)).mean() * 100

        history['train_loss'].append(running_loss / len(train_loader))
        history['val_loss'].append(val_running_loss / len(val_loader))
        history['val_acc'].append(acc)
        history['val_uar'].append(uar)

    return history

# Thực thi K-Fold nếu có dữ liệu
if os.path.exists(features_dir) and len(os.listdir(features_dir)) > 0:
    dataset = EmotionDataset(features_dir)
    kfold_history = train_with_kfold(dataset, n_splits=5, epochs=100)
else:
    print(f"❌ Lỗi: Thư mục {features_dir} đang trống hoặc không tồn tại.")
    print("💡 Hãy chạy cell trích xuất đặc trưng (Wav2Vec) trước khi huấn luyện.")

In [ ]:
import numpy as np

def report_final_kfold_results(kfold_history):
    final_uars = []
    final_accs = []

    print("\n=== 📊 TỔNG HỢP KẾT QUẢ K-FOLD (5 FOLDS) ===")
    for i, history in enumerate(kfold_history):
        # Lấy giá trị tốt nhất từ mỗi fold
        best_fold_uar = max(history['val_uar'])
        best_fold_acc = max(history['val_acc'])
        final_uars.append(best_fold_uar)
        final_accs.append(best_fold_acc)
        print(f"Fold {i+1}: Best Val UAR = {best_fold_uar:.2f}%, Best Val Acc = {best_fold_acc:.2f}%")

    mean_uar = np.mean(final_uars)
    std_uar = np.std(final_uars)
    mean_acc = np.mean(final_accs)

    print("\n--- 🏆 KẾT QUẢ CUỐI CÙNG ---")
    print(f"Mean Validation UAR: {mean_uar:.2f}% (+/- {std_uar:.2f}%)")
    print(f"Mean Validation Accuracy: {mean_acc:.2f}%")
    print("--------------------------")

# Kiểm tra sự tồn tại của dữ liệu trước khi chạy
if 'kfold_history' in globals() or 'kfold_history' in locals():
    report_final_kfold_results(kfold_history)
else:
    print("❌ Lỗi: Không tìm thấy 'kfold_history'.")
    print("💡 Hướng dẫn: Bạn cần chạy thành công cell 'Huấn luyện với K-Fold' (cell 2f9ed861) trước khi xem báo cáo này.")

### 📊 Hiển thị Ma trận nhầm lẫn (Confusion Matrix)
Bước này giúp chúng ta nhìn rõ mô hình đang phân loại đúng/sai ở những nhãn nào cụ thể trong 8 lớp cảm xúc.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix
import numpy as np

def plot_confusion_matrix(model, val_loader):
    model.eval()
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for features, labels, lengths in val_loader:
            features, labels, lengths = features.to(device), labels.to(device), lengths.to(device)
            logits, _ = model(features, lengths)
            _, predicted = logits.max(1)

            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    # Định nghĩa nhãn cảm xúc
    emotion_labels = ['Neutral', 'Calm', 'Happy', 'Sad', 'Angry', 'Fearful', 'Disgust', 'Surprised']

    # Tính toán ma trận
    cm = confusion_matrix(all_labels, all_preds, labels=range(8))

    # Vẽ biểu đồ
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=emotion_labels,
                yticklabels=emotion_labels)
    plt.xlabel('Dự đoán (Predicted)')
    plt.ylabel('Thực tế (Actual)')
    plt.title('Ma trận nhầm lẫn cho 8 lớp cảm xúc')
    plt.show()

# Chạy hiển thị ma trận sau khi đã huấn luyện xong
if 'model' in locals():
    plot_confusion_matrix(model, val_loader)
else:
    print("⚠️ Biến 'model' chưa được định nghĩa.")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix
import numpy as np
import torch

def plot_final_confusion_matrix(model, loader):
    model.eval()
    y_true = []
    y_pred = []

    with torch.no_grad():
        for features, labels, lengths in loader:
            features, labels, lengths = features.to(device), labels.to(device), lengths.to(device)
            logits, _ = model(features, lengths)
            _, predicted = torch.max(logits, 1)

            y_true.extend(labels.cpu().numpy())
            y_pred.extend(predicted.cpu().numpy())

    emotion_labels = ['Neutral', 'Calm', 'Happy', 'Sad', 'Angry', 'Fearful', 'Disgust', 'Surprised']
    cm = confusion_matrix(y_true, y_pred, labels=range(8))

    plt.figure(figsize=(12, 9))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=emotion_labels,
                yticklabels=emotion_labels)
    plt.xlabel('Dự đoán (Predicted)')
    plt.ylabel('Thực tế (Actual)')
    plt.title('Ma trận nhầm lẫn - Phân loại cảm xúc (8 lớp)')
    plt.show()

# Thực thi hiển thị
if 'model' in locals():
    plot_final_confusion_matrix(model, val_loader)
else:
    print("❌ Không tìm thấy biến 'model'. Vui lòng chạy lại các bước huấn luyện trước đó.")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix
import torch
import numpy as np

# Ensure model is in eval mode
model.eval()
y_true = []
y_pred = []

print("📊 Calculating confusion matrix for 8 classes on Validation set...")
with torch.no_grad():
    for features, labels, lengths in val_loader:
        features = features.to(device)
        labels = labels.to(device)
        lengths = lengths.to(device)

        logits, _ = model(features, lengths)
        _, predicted = torch.max(logits, 1)

        y_true.extend(labels.cpu().numpy())
        y_pred.extend(predicted.cpu().numpy())

emotion_labels = ['Neutral', 'Calm', 'Happy', 'Sad', 'Angry', 'Fearful', 'Disgust', 'Surprised']
cm = confusion_matrix(y_true, y_pred, labels=range(8))

plt.figure(figsize=(12, 10))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=emotion_labels,
            yticklabels=emotion_labels)
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.title('Confusion Matrix: ResLSTM-SER (8 Synthetic Classes - 100 Epochs)')
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix
import numpy as np

# Sử dụng kết quả y_true và y_pred đã tính ở ô trước
emotion_labels = ['Neutral', 'Calm', 'Happy', 'Sad', 'Angry', 'Fearful', 'Disgust', 'Surprised']
cm = confusion_matrix(y_true, y_pred, labels=range(8))

# Hiển thị biểu đồ
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=emotion_labels,
            yticklabels=emotion_labels)
plt.xlabel('Dự đoán (Predicted)')
plt.ylabel('Thực tế (Actual)')
plt.title('Ma trận nhầm lẫn - Phân loại cảm xúc (ResLSTM)')
plt.show()

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix
import numpy as np

# Sử dụng lại y_true và y_pred đã có sẵn trong bộ nhớ
emotion_labels = ['Neutral', 'Calm', 'Happy', 'Sad', 'Angry', 'Fearful', 'Disgust', 'Surprised']
cm = confusion_matrix(y_true, y_pred, labels=range(8))

# Khởi tạo và vẽ lại
plt.figure(figsize=(10, 8))
sns.set(style='white')
ax = sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                 xticklabels=emotion_labels,
                 yticklabels=emotion_labels)

plt.xlabel('Dự đoán (Predicted)', fontsize=12)
plt.ylabel('Thực tế (Actual)', fontsize=12)
plt.title('Ma trận nhầm lẫn - Emotion Classification (ResLSTM)', fontsize=14)
plt.show()

In [ ]:
from transformers import Wav2Vec2Model, Wav2Vec2Processor
import torch

print("⌛ Đang nေ̣p Wav2Vec 2.0 Large (Facebook/wav2vec2-large-960h)...")
model_name = "facebook/wav2vec2-large-960h"

try:
    processor = Wav2Vec2Processor.from_pretrained(model_name)
    wav2vec_model = Wav2Vec2Model.from_pretrained(model_name).to(device)
    wav2vec_model.eval()
    print("✅ Đẵ nေ̣p Wav2Vec 2.0 Large l ̂n GPU!")
except Exception as e:
    print(f"❌ Lᄑ̃i nေ̣p model: {e}")

# Soft Speech Units for Improved Voice Conversion

Demo for the paper: [A Comparison of Discrete and Soft Speech Units for Improved Voice Conversion](https://ieeexplore.ieee.org/abstract/document/9746484).

- [Companion webpage](https://bshall.github.io/soft-vc/)
- [Home repo](https://github.com/bshall/soft-vc)
- [HuBERT content encoders](https://github.com/bshall/hubert)
- [Acoustic Models](https://github.com/bshall/acoustic-model)
- [HiFiGAN vocoder](https://github.com/bshall/hifigan)

In [ ]:
import sys
import os

# 1. Sử dụng đường dẫn tuyệt đối bắt đầu bằng /content/ để tránh lạc hướng
src_path = "/content/ResLSTM-SER/src"

# 2. Thêm vào hệ thống nếu đường dẫn tồn tại
if os.path.exists(src_path):
    if src_path not in sys.path:
        sys.path.append(src_path)
    print(f"🔗 Đã ép kết nối tuyệt đối tới thư mục: {src_path}")
else:
    print(f"❌ Cảnh báo: Không tìm thấy thư mục {src_path} trên máy ảo!")

# 3. Tiến hành kiểm tra Import lại
try:
    from res_lstm import ResLSTM_Multi_Att, MODEL_VERSION
    print("✅ Kiểm tra Import THÀNH CÔNG!")
    print(f"🤖 Mô hình thế hệ mới: ResLSTM_Multi_Att ({MODEL_VERSION})")
except ImportError as e:
    print(f"❌ Vẫn gặp lỗi Import: {e}")

In [ ]:
import sys
import os

# 1. Ép đường dẫn tuyệt đối đi thẳng vào thư mục models
src_path = "/content/ResLSTM-SER/src/models"

# 2. Kết nối vào hệ thống Python của Colab
if os.path.exists(src_path):
    if src_path not in sys.path:
        sys.path.append(src_path)
    print(f"🔗 Đã ép kết nối tuyệt đối tới thư mục con chứa mô hình: {src_path}")
else:
    print(f"❌ Cảnh báo: Không tìm thấy đường dẫn {src_path}")

# 3. Tiến hành kiểm tra Import lại từ file res_lstm.py
try:
    from res_lstm import ResLSTM_Multi_Att, MODEL_VERSION
    print("✅ Kiểm tra Import THÀNH CÔNG!")
    print(f"🤖 Mô hình thế hệ mới đã sẵn sàng: ResLSTM_Multi_Att ({MODEL_VERSION})")
except ImportError as e:
    print(f"❌ Vẫn gặp lỗi Import: {e}. Hãy chạy lệnh !find /content/ -name 'res_lstm.py' để định vị.")

In [ ]:
import torch

# 1. Giả định một Batch dữ liệu đầu vào trích xuất từ bộ lọc Wav2Vec 2.0 Large
# Kích thước ma trận: (Batch_size=4, Time_steps=50, Input_size=1024)
X_dummy = torch.randn(4, 50, 1024)

# Độ dài thực tế của từng câu âm thanh trong Batch (tránh các đoạn padding 0)
lengths_dummy = torch.tensor([50, 45, 40, 35], dtype=torch.long)

# 2. Khởi tạo thử mô hình thế hệ mới với tham số chuẩn của bạn
device = "cuda" if torch.cuda.is_available() else "cpu"
model_test = ResLSTM_Multi_Att(
    input_size=1024,   # 1024 chiều để gánh vector ngữ cảnh sâu từ Bài 3 thay vì 46 chiều MFCC cũ
    hidden_size=64,    # Tham số ẩn tối ưu của bạn
    num_layers=1,      # Cấu trúc tầng của LSTM
    num_att=4,         # Sử dụng 4 đầu đa chú ý (Multi-vector Attention Variant A) để lọc nhiễu dữ liệu nhân tạo
    num_classes=8,     # 8 lớp cảm xúc phân loại của tập dữ liệu RAVDESS
    device=device
).to(device)

# 3. Chạy thử luồng Forward Pass
model_test.eval()
with torch.no_grad():
    logits, att_weights = model_test(X_dummy.to(device), lengths_dummy.to(device))

print("--------------------------------------------------")
print(f"🔮 Kết quả kích thước Logits đầu ra: {logits.shape} -> Khớp chuẩn [Batch, Classes]!")
print(f"📐 Kích thước trọng số Attention thu được: {att_weights.shape} -> Sẵn sàng làm gác cổng lọc nhiễu!")
print("--------------------------------------------------")
print("🎉 Cấu trúc mạng hoạt động hoàn hảo 100%!")

In [ ]:
import os
import glob
from google.colab import files

# 1. Tìm kiếm mở rộng trên toàn bộ Drive
print("🔍 Đang quét toàn bộ Drive để tìm 'archive.zip'...")
search_pattern = "/content/drive/MyDrive/**/archive.zip"
found_zips = glob.glob(search_pattern, recursive=True)

extract_path = "./data/ravdess"
os.makedirs(extract_path, exist_ok=True)

if found_zips:
    zip_path = found_zips[0]
    print(f"📦 Đã tìm thấy file tại: {zip_path}")
    !unzip -q "{zip_path}" -d {extract_path}
    audio_files = glob.glob(f"{extract_path}/**/*.wav", recursive=True)
    print(f"✅ Thành công! Đã giải nén {len(audio_files)} file.")
else:
    print("❌ Không tìm thấy 'archive.zip' trên Drive.")
    print("👇 TÙY CHỌN: Bạn có thể upload trực tiếp file dataset từ máy tính lên đây:")
    uploaded = files.upload()
    for fn in uploaded.keys():
        if fn.endswith('.zip'):
            !unzip -q "{fn}" -d {extract_path}
            audio_files = glob.glob(f"{extract_path}/**/*.wav", recursive=True)
            print(f"✅ Đã giải nén từ file vừa upload: {len(audio_files)} file.")

🔍 Đang quét toàn bộ Drive để tìm 'archive.zip'...
❌ Không tìm thấy 'archive.zip' trên Drive.
👇 TÙY CHỌN: Bạn có thể upload trực tiếp file dataset từ máy tính lên đây:


In [17]:
import os
import glob

# Đường dẫn gốc dự án trên Drive
project_root = "/content/drive/MyDrive/ResLSTM-SER"
extract_path = "./data/ravdess"

os.makedirs(extract_path, exist_ok=True)

print(f"📂 Đang tìm kiếm file zip trong: {project_root}")

# Tìm tất cả file .zip trong thư mục dự án
zips = glob.glob(f"{project_root}/**/*.zip", recursive=True)

if zips:
    print(f"✅ Tìm thấy {len(zips)} file zip:")
    for z in zips:
        print(f" - {z}")

    # Ưu tiên tìm file có tên archive.zip hoặc các file zip chứa dataset
    target_zip = next((z for z in zips if "archive" in z.lower()), zips[0])

    print(f"⏳ Đang giải nén file: {target_zip}")
    !unzip -q "{target_zip}" -d {extract_path}

    audio_files = glob.glob(f"{extract_path}/**/*.wav", recursive=True)
    print(f"🎉 Hoàn thành! Đã giải nén được {len(audio_files)} file âm thanh.")
else:
    print("❌ Vẫn không tìm thấy file .zip nào trong thư mục ResLSTM-SER.")
    print("Nội dung hiện tại của thư mục dự án:")
    !ls -R "{project_root}"

📂 Đang tìm kiếm file zip trong: /content/drive/MyDrive/ResLSTM-SER
✅ Tìm thấy 2 file zip:
 - /content/drive/MyDrive/ResLSTM-SER/ResLTSMv2.zip
 - /content/drive/MyDrive/ResLSTM-SER/data/archive.zip
⏳ Đang giải nén file: /content/drive/MyDrive/ResLSTM-SER/data/archive.zip
🎉 Hoàn thành! Đã giải nén được 2880 file âm thanh.


In [18]:
import glob
import os

# Đếm chính xác số lượng file .wav trong thư mục ravdess
wav_files = glob.glob("./data/ravdess/**/*.wav", recursive=True)
print(f"🔍 Số lượng file .wav thực tế tìm thấy: {len(wav_files)}")

# Hiển thị cấu trúc thư mục con để kiểm tra xem có bị trùng lặp không
actor_folders = sorted(glob.glob("./data/ravdess/*"))
print(f"📁 Số lượng thư mục Actor: {len([d for d in actor_folders if os.path.isdir(d)])}")

🔍 Số lượng file .wav thực tế tìm thấy: 2880
📁 Số lượng thư mục Actor: 25


In [22]:
# 1. Di chuyển tất cả Actor_xx từ thư mục con ra ngoài thư mục gốc ./data/ravdess
!find ./data/ravdess -type d -name "Actor_*" -exec mv -t ./data/ravdess/ {} + 2>/dev/null

# 2. Xóa thư mục cha trống hoặc các file thừa
!rm -rf ./data/ravdess/audio_speech_actors_01-24
!find ./data/ravdess -name "*._*" -delete
!find ./data/ravdess -name ".DS_Store" -delete

import glob
import os
# Kiểm tra lại đường dẫn Actor trực tiếp
actors = sorted(glob.glob("./data/ravdess/Actor_*"))
wav_count = len(glob.glob("./data/ravdess/Actor_*/*.wav"))

print(f"✅ Cấu trúc đã được chuẩn hóa!")
print(f"📁 Số lượng thư mục Actor: {len(actors)} (Từ {os.path.basename(actors[0]) if actors else 'N/A'} đến {os.path.basename(actors[-1]) if actors else 'N/A'})")
print(f"🎵 Tổng số file .wav: {wav_count}")

✅ Cấu trúc đã được chuẩn hóa!
📁 Số lượng thư mục Actor: 24 (Từ Actor_01 đến Actor_24)
🎵 Tổng số file .wav: 1440


In [23]:
import os
import glob

print("📊 KIỂM TRA CẤU TRÚC CUỐI CÙNG:")
base_dir = './data/ravdess'
actors = sorted([d for d in os.listdir(base_dir) if os.path.isdir(os.path.join(base_dir, d)) and 'Actor_' in d])

print(f"✅ Tìm thấy {len(actors)} thư mục Actor.")
for actor in actors[:3]: # Hiển thị mẫu 3 actor đầu
    count = len(glob.glob(f'{base_dir}/{actor}/*.wav'))
    print(f"   - {actor}: {count} file .wav")
print("   ...")

total_wavs = len(glob.glob(f'{base_dir}/Actor_*/*.wav'))
print(f"🚀 TỔNG CỘNG: {total_wavs} file .wav (Chuẩn RAVDESS: 1440)")

📊 KIỂM TRA CẤU TRÚC CUỐI CÙNG:
✅ Tìm thấy 24 thư mục Actor.
   - Actor_01: 60 file .wav
   - Actor_02: 60 file .wav
   - Actor_03: 60 file .wav
   ...
🚀 TỔNG CỘNG: 1440 file .wav (Chuẩn RAVDESS: 1440)


In [21]:
import os

def list_files(startpath):
    print(f"📂 Cấu trúc thư mục hiện tại của {startpath}:")
    for root, dirs, files in os.walk(startpath):
        level = root.replace(startpath, '').count(os.sep)
        indent = ' ' * 4 * (level)
        print(f"{indent}📁 {os.path.basename(root)}/ ({len(files)} files)")
        # Chỉ hiện 1 file mẫu để tránh làm rối màn hình
        if files:
            subindent = ' ' * 4 * (level + 1)
            print(f"{subindent}📄 {files[0]} ...")

list_files('./data/ravdess')

📂 Cấu trúc thư mục hiện tại của ./data/ravdess:
📁 ravdess/ (0 files)
    📁 Actor_19/ (60 files)
        📄 03-01-06-02-01-01-19.wav ...
    📁 Actor_18/ (60 files)
        📄 03-01-02-01-01-02-18.wav ...
    📁 Actor_07/ (60 files)
        📄 03-01-06-01-01-01-07.wav ...
    📁 Actor_17/ (60 files)
        📄 03-01-05-02-01-01-17.wav ...
    📁 Actor_06/ (60 files)
        📄 03-01-06-02-02-02-06.wav ...
    📁 audio_speech_actors_01-24/ (0 files)
        📁 Actor_19/ (60 files)
            📄 03-01-06-02-01-01-19.wav ...
        📁 Actor_18/ (60 files)
            📄 03-01-02-01-01-02-18.wav ...
        📁 Actor_07/ (60 files)
            📄 03-01-06-01-01-01-07.wav ...
        📁 Actor_17/ (60 files)
            📄 03-01-05-02-01-01-17.wav ...
        📁 Actor_06/ (60 files)
            📄 03-01-06-02-02-02-06.wav ...
        📁 Actor_22/ (60 files)
            📄 03-01-04-02-02-02-22.wav ...
        📁 Actor_24/ (60 files)
            📄 03-01-02-01-01-02-24.wav ...
        📁 Actor_14/ (60 files)
         

In [15]:
from google.colab import drive
import os

print("🔄 Đang tiến hành kết nối lại Google Drive...")
try:
    # Buộc mount lại
    drive.mount('/content/drive', force_remount=True)

    if os.path.exists('/content/drive/MyDrive'):
        print("✅ Kết nối Drive THÀNH CÔNG!")
        print("📂 Danh sách thư mục gốc trên Drive của bạn:")
        folders = [f for f in os.listdir('/content/drive/MyDrive') if os.path.isdir(os.path.join('/content/drive/MyDrive', f))]
        for f in sorted(folders):
            print(f" - {f}")
    else:
        print("❌ Mount thành công nhưng không tìm thấy thư mục MyDrive.")
except Exception as e:
    print(f"❌ Lỗi kết nối Drive: {e}")
    print("💡 Vui lòng bấm vào biểu tượng Thư mục ở menu bên trái, sau đó bấm vào biểu tượng 'Mount Drive' (hình thư mục có logo Drive).")

🔄 Đang tiến hành kết nối lại Google Drive...
Mounted at /content/drive
✅ Kết nối Drive THÀNH CÔNG!
📂 Danh sách thư mục gốc trên Drive của bạn:
 - 24110148_duongduyvinh_lab1
 - Colab Notebooks
 - Nhóm 4 - C2_S3_DIPR430685E_01FIE
 - Presentation_DuongDuyVinh_24110148
 - ResLSTM-SER
 - Speech Processing
 - programming technique


In [16]:
from google.colab import drive
import os

print("🔄 Đang tiến hành kết nối lại Google Drive...")
try:
    # Buộc mount lại
    drive.mount('/content/drive', force_remount=True)

    if os.path.exists('/content/drive/MyDrive'):
        print("✅ Kết nối Drive THÀNH CÔNG!")
        print("📂 Danh sách thư mục gốc trên Drive của bạn:")
        folders = [f for f in os.listdir('/content/drive/MyDrive') if os.path.isdir(os.path.join('/content/drive/MyDrive', f))]
        for f in sorted(folders):
            print(f" - {f}")
    else:
        print("❌ Mount thành công nhưng không tìm thấy thư mục MyDrive.")
except Exception as e:
    print(f"❌ Lỗi kết nối Drive: {e}")
    print("💡 Vui lòng bấm vào biểu tượng Thư mục ở menu bên trái, sau đó bấm vào biểu tượng 'Mount Drive' (hình thư mục có logo Drive).")

🔄 Đang tiến hành kết nối lại Google Drive...
Mounted at /content/drive
✅ Kết nối Drive THÀNH CÔNG!
📂 Danh sách thư mục gốc trên Drive của bạn:
 - 24110148_duongduyvinh_lab1
 - Colab Notebooks
 - Nhóm 4 - C2_S3_DIPR430685E_01FIE
 - Presentation_DuongDuyVinh_24110148
 - ResLSTM-SER
 - Speech Processing
 - programming technique


In [ ]:
import os
import torch
import torchaudio

# 1. Định nghĩa đường dẫn file nguồn và file đích (RAVDESS)
source_path = "./data/ravdess/audio_speech_actors_01-24/Actor_01/03-01-01-01-01-01-01.wav"
target_path = "./data/ravdess/audio_speech_actors_01-24/Actor_01/03-01-05-01-01-01-01.wav"

# Tạo thư mục lưu trên Google Drive vĩnh viễn
output_dir = "/content/drive/MyDrive/ResLSTM-SER/data/synthetic_wavs"
os.makedirs(output_dir, exist_ok=True)
output_path = os.path.join(output_dir, "syn_actor01_neutral_to_angry.wav")

try:
    print("⏳ Đang xử lý bẻ giọng bằng mô hình Soft-VC...")

    # Load audio thô
    source_waveform, sr1 = torchaudio.load(source_path)
    target_waveform, sr2 = torchaudio.load(target_path)

    # Kiểm tra và lấy mô hình thực sự từ tuple hubert
    # Nếu biến hubert là một tuple, ta lấy phần tử đầu tiên chính là thực thể mô hình
    hubert_model = hubert[0] if isinstance(hubert, tuple) else hubert
    hifigan_model = hifigan[0] if isinstance(hifigan, tuple) else hifigan

    # Đảm bảo đưa mô hình về chế độ Evaluation và chuyển sang GPU
    hubert_model.eval().cuda()
    hifigan_model.eval().cuda()

    # Chuẩn bị tensor đầu vào cho GPU (Format: Batch x Channels x Time)
    x_source = source_waveform.cuda()
    x_target = target_waveform.cuda()

    with torch.no_grad():
        # Bước A: Sử dụng thực thể mô hình đã trích xuất để lấy Soft Units
        units = hubert_model.extract_features(x_source)

        # Bước B: Ép sắc thái cảm xúc từ target qua bộ tổng hợp Vocoder
        # Lưu ý: Tùy phiên bản repo soft-vc, hàm tổng hợp có tên là .syn() hoặc .forward()
        if hasattr(hifigan_model, 'syn'):
            converted_waveform = hifigan_model.syn(units, x_target)
        else:
            converted_waveform = hifigan_model(units, x_target)

    # Lưu file nhân tạo thu được trực tiếp lên Google Drive vĩnh viễn
    torchaudio.save(output_path, converted_waveform.squeeze(0).cpu(), sample_rate=16000)
    print("--------------------------------------------------")
    print(f"✅ BẺ GIỌNG THÀNH CÔNG! File dữ liệu nhân tạo đầu tiên đã được cất tại:")
    print(output_path)
    print("--------------------------------------------------")

except Exception as e:
    print(f"❌ Gặp lỗi hệ thống: {e}")
    print("💡 Mẹo: Nếu gặp lỗi kích thước Tensor, hãy đảm bảo các file .wav đầu vào đã được resample về cùng tần số (thường là 16000Hz).")

In [ ]:
import torchaudio
import torch

def convert_voice_safe(source_path, target_path, output_path):
    try:
        # Kiểm tra thiết bị hiện tại (đã định nghĩa ở cell trước là 'cpu' hoặc 'cuda')
        device = "cuda" if torch.cuda.is_available() else "cpu"

        # 1. Load và resample source audio về 16kHz
        source, sr = torchaudio.load(source_path)
        source = torchaudio.functional.resample(source, sr, 16000)
        if source.shape[0] > 1: source = source.mean(0, keepdim=True)
        source = source.unsqueeze(0).to(device)

        # 2. Xử lý qua bộ 3 model Soft-VC
        with torch.inference_mode():
            # Trích xuất đơn vị âm thanh
            units = hubert.units(source)
            # Tạo phổ spectrogram (với đặc điểm giọng của target - ở đây dùng chính nó để test)
            mel = acoustic.generate(units).transpose(1, 2)
            # Chuyển phổ thành sóng âm thanh
            target_out = hifigan(mel)

        # 3. Lưu kết quả ra file
        torchaudio.save(output_path, target_out.squeeze(0).cpu(), 16000)
        print(f"✅ Đã tạo file thành công: {output_path}")
    except Exception as e:
        print(f"❌ Lỗi: {e}")

# Thử nghiệm bẻ giọng với một file từ tập RAVDESS bạn đã giải nén
sample_source = "./data/ravdess/audio_speech_actors_01-24/Actor_01/03-01-01-01-01-01-01.wav"
if os.path.exists(sample_source):
    convert_voice_safe(sample_source, sample_source, "test_converted.wav")
else:
    # Nếu chưa có data RAVDESS thì dùng file example
    convert_voice_safe("example.wav", "example.wav", "test_converted.wav")